Complete code for processing and plotting daily polar maps of ozone column anomaly.
Each image is annotated with:
  1. The reference (year 008) ozone column (area average) on that day.
  2. The simulation (J/F/M) ozone column (area average) on that day.
  
The area‐average is computed as in your original code:
  (average over longitude) then (cosine–weighted average over latitude).
  
Adjust the file paths and parameters as needed.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

#########################################
# Data Processing Functions
#########################################

In [ ]:
def calc_parc_o3(var, p_top=30, p_bottom=100):
    """
    Calculate the partial ozone column for a specified pressure range.

    Parameters:
        var: xarray.DataArray
            Ozone data (units: mol/mol)
        p_top: float
            Top pressure level (hPa)
        p_bottom: float
            Bottom pressure level (hPa)

    Returns:
        xarray.DataArray:
            Ozone column in Dobson Units (DU)
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    # Determine pressure coordinate name (plev, lev, or level)
    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    elif 'level' in var.dims:
        plev = var.level
        level = 'level'
    else:
        raise ValueError("Pressure coordinate not found: plev/lev/level")

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # Determine the direction and conversion factors for pressure coordinate
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100  # convert hPa to Pa
        factor_2 = 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1
        factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100
    else:
        factor = 1
        factor_2 = 1

    # If pressure levels are in ascending order
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        # Select layers between p_top and p_bottom
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16  # convert to DU
    else:
        for i in range(len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16  # convert to DU

    return O3.where(O3 != 0)

def load_group_data(file_pattern, p_top=30, p_bottom=70, lat_min=60, lat_max=90):
    """
    Load and process ozone data for a simulation group (e.g., J, F, or M).

    Parameters:
        file_pattern: str
            File path pattern (supports wildcards), e.g.,
            '.../Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
        p_top, p_bottom: float
            Pressure range in hPa.
        lat_min, lat_max: float
            Latitude range (in degrees).

    Returns:
        xarray.DataArray:
            Ensemble-averaged ozone column with time, lat, and lon coordinates.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    o3 = ds['O3']
    o3_col = calc_parc_o3(o3, p_top, p_bottom)
    o3_col = o3_col.sel(lat=slice(lat_min, lat_max))
    if 'member' in o3_col.dims:
        o3_mean = o3_col.mean(dim='member')
    else:
        o3_mean = o3_col
    return o3_mean

def compute_area_average(field):
    """
    Compute the area-average of a field over lat and lon.
    
    If the field has a 'lon' dimension with more than one value, first average over lon,
    then compute the cosine–weighted average over lat.
    """
    if 'lon' in field.dims and field.lon.size > 1:
        field_avg = field.mean(dim='lon')
    else:
        field_avg = field
    if 'lat' in field_avg.dims:
        weights = np.cos(np.deg2rad(field_avg.lat))
        field_avg = field_avg.weighted(weights).mean(dim='lat')
    return field_avg

#########################################

Plotting Function

#########################################

In [ ]:
def plot_daily_maps(group_name, group_data, ref_data, output_dir, day_offset=0):
    """
    Plot daily polar maps of the ozone column anomaly and overlay area-average values.

    The anomaly is computed as:
      anomaly = (simulation ensemble mean) - (reference 008 data at the corresponding time)
    
    In addition, the following information is added on the plot:
      1. Reference ozone column (area-average) for that day.
      2. Simulation (J/F/M) ozone column (area-average) for that day.
    
    Parameters:
        group_name: str
            Group name (e.g., 'J').
        group_data: xarray.DataArray
            Ensemble-averaged ozone column (with time, lat, lon) in DU.
        ref_data: xarray.DataArray
            Reference ozone data (processed and subset to 60–90°N).
        output_dir: str
            Directory to save output images.
        day_offset: int
            Time offset for the reference data (default is 0).
    """
    times = group_data.time.values
    n_days = len(times)
    os.makedirs(output_dir, exist_ok=True)

    rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
    rc('text', usetex=False)

    for i in range(n_days):
        ref_index = day_offset + i
        if ref_index >= ref_data.time.size:
            print(f"No reference data for day_offset={ref_index}, skipping {group_name} day {i+1}.")
            continue

        sim_field = group_data.isel(time=i)
        ref_field = ref_data.isel(time=ref_index)
        # If reference data has only one longitude, broadcast it
        if ('lon' in ref_field.dims) and (ref_field.lon.size == 1):
            ref_field = ref_field.broadcast_like(sim_field)

        anomaly = sim_field - ref_field

        # Compute area averages (the same as your original calculation)
        sim_avg = float(compute_area_average(sim_field).compute())
        ref_avg = float(compute_area_average(ref_field).compute())

        # Add cyclic point to avoid blank wedge between 359° and 0°
        anomaly_cyclic, cyclic_lons = add_cyclic_point(anomaly.values,
                                                       coord=anomaly.lon.values)

        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo(central_longitude=0))
        ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())

        levels = np.linspace(-50, 50, 21)
        cf = ax.contourf(cyclic_lons, anomaly.lat, anomaly_cyclic,
                         levels=levels,
                         cmap='RdBu_r', extend='both', transform=ccrs.PlateCarree())

        ax.coastlines()
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')

        cbar = fig.colorbar(cf, ax=ax, orientation='horizontal', fraction=0.04, pad=0.08)
        cbar.set_label("Ozone Column Anomaly (DU)", fontsize=12)

        # Overlay the area-average values on the plot
        text_str = f"Reference O3: {ref_avg:.2f} DU\n{group_name} O3: {sim_avg:.2f} DU"
        ax.text(0.02, 0.98, text_str, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

        # --- Fix for time label ---
        # Get the time value, ensuring it is a NumPy datetime64 scalar.
        time_val = sim_field.time.values
        if hasattr(time_val, 'compute'):
            time_val = time_val.compute()
        if isinstance(time_val, np.ndarray) and time_val.size == 1:
            time_val = time_val.item()
        time_label = group_map.time.dt.strftime('%Y-%m-%d').values.item()
        # ---------------------------

        ax.set_title(f"{group_name} Group, Day {i+1:02d}\nSimulation Date: {time_label}",
                     fontsize=14)

        out_fname = os.path.join(output_dir, f"{group_name}_day_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150)
        plt.close()
        print(f"Saved {group_name} group day {i+1} image to {out_fname}")


#########################################
# Main Data Processing Section
#########################################

In [ ]:
# Load reference data from file and filter to year 008 and months January–May.
ref_path = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
ds_ref = xr.open_dataset(ref_path)
ref_o3 = ds_ref['O3']
ref_col = calc_parc_o3(ref_o3, p_top=30, p_bottom=70)
ref_col = ref_col.sel(lat=slice(60, 90))
if 'member' in ref_col.dims:
    ref_mean = ref_col.mean(dim='member')
else:
    ref_mean = ref_col

# Filter the reference data to year 008 and months January–May.
ref_mean = ref_mean.sel(time=(ref_mean.time.dt.year == 8) &
                        (ref_mean.time.dt.month.isin([1, 2, 3, 4, 5]))).sortby('time')

# Define the months to use for each simulation group.
group_months = {
    'J': [1, 2, 3, 4, 5],
    'F': [2, 3, 4, 5],
    'M': [3, 4, 5]
}

# Define simulation groups with file patterns and output directories.
# Output directories are set to "./plots/2d/<group>"
groups = {
    'J': {
        'file_pattern': "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc",
        'output_dir': "./plots/2d/J"
    },
    'F': {
        'file_pattern': "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc",
        'output_dir': "./plots/2d/F"
    },
    'M': {
        'file_pattern': "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc",
        'output_dir': "./plots/2d/M"
    }
}

# Process each simulation group and store the processed data.
simulation_data = {}
reference_data = {}
for group_name, info in groups.items():
    print(f"Processing group {group_name} data...")
    group_data = load_group_data(info['file_pattern'], p_top=30, p_bottom=70, lat_min=60, lat_max=90)
    # Filter simulation data to the appropriate months
    group_data = group_data.sel(time=group_data.time.dt.month.isin(group_months[group_name])).sortby('time')
    simulation_data[group_name] = group_data

    # Also filter the reference data for the corresponding months.
    ref_subset = ref_mean.sel(time=ref_mean.time.dt.month.isin(group_months[group_name])).sortby('time')
    reference_data[group_name] = ref_subset

#########################################
# Plotting Section
#########################################

In [ ]:
for group_name in groups.keys():
    plot_daily_maps(group_name,
                    simulation_data[group_name],
                    reference_data[group_name],
                    groups[group_name]['output_dir'],
                    day_offset=0)

print("All plotting completed!")